<h1 style="text-align:center; font-size:42px; margin-top:40px;">
Suppression des doublons – Stack Overflow Developer Survey 2024
</h1>



## 1. Contexte et objectif

L’objectif de ce module est de produire une version **dédoublonnée** du dataset Stack Overflow Developer Survey 2024.

Plus précisément, ce notebook :

- sélectionne les colonnes pertinentes pour l’analyse des tendances technologiques (langages, bases de données, plateformes, frameworks, etc.) ;
- définit un ensemble de **colonnes clé** décrivant le profil d’un développeur ;
- identifie et supprime les **lignes dupliquées** sur la base de ces colonnes clé ;
- génère un dataset propre, sans doublons logiques, qui servira d’entrée à l’étape suivante : **traitement des valeurs manquantes**.

Ce module s’inscrit dans le pipeline global après l’exploration initiale et avant le nettoyage approfondi / EDA.

## 2. Librairies et chargement des données

In [2]:
import pandas as pd

# Chargement du dataset brut (version V0)
df = pd.read_csv("../Data/Survey/raw/survey-data_V0.csv")

# Aperçu
df.head()


,ResponseId,MainBranch,Age,Employment,RemoteWork,Check,CodingActivities,EdLevel,LearnCode,LearnCodeOnline,...,JobSatPoints_6,JobSatPoints_7,JobSatPoints_8,JobSatPoints_9,JobSatPoints_10,JobSatPoints_11,SurveyLength,SurveyEase,ConvertedCompYearly,JobSat
0,1,I am a developer by profession,Under 18 years old,"Employed, full-time",Remote,Apples,Hobby,Primary/elementary school,Books / Physical media,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,I am a developer by profession,35-44 years old,"Employed, full-time",Remote,Apples,Hobby;Contribute to open-source projects;Other...,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Books / Physical media;Colleague;On the job tr...,Technical documentation;Blogs;Books;Written Tu...,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
2,3,I am a developer by profession,45-54 years old,"Employed, full-time",Remote,Apples,Hobby;Contribute to open-source projects;Other...,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Books / Physical media;Colleague;On the job tr...,Technical documentation;Blogs;Books;Written Tu...,...,NaN,NaN,NaN,NaN,NaN,NaN,Appropriate in length,Easy,NaN,NaN
3,4,I am learning to code,18-24 years old,"Student, full-time",NaN,Apples,NaN,Some college/university study without earning ...,"Other online resources (e.g., videos, blogs, f...",Stack Overflow;How-to videos;Interactive tutorial,...,NaN,NaN,NaN,NaN,NaN,NaN,Too long,Easy,NaN,NaN
4,5,I am a developer by profession,18-24 years old,"Student, full-time",NaN,Apples,NaN,"Secondary school (e.g. American high school, G...","Other online resources (e.g., videos, blogs, f...",Technical documentation;Blogs;Written Tutorial...,...,NaN,NaN,NaN,NaN,NaN,NaN,Too short,Easy,NaN,NaN


## 3. Types de données

In [3]:
print(df.dtypes.value_counts())
print("------------")
print(df.dtypes)

object     100
float64     13
int64        1
Name: count, dtype: int64
------------
ResponseId               int64
MainBranch              object
Age                     object
Employment              object
RemoteWork              object
                        ...   
JobSatPoints_11        float64
SurveyLength            object
SurveyEase              object
ConvertedCompYearly    float64
JobSat                 float64
Length: 114, dtype: object


## 4. Sélection des colonnes pertinentes pour l’analyse

Le dataset d’origine contient 114 colonnes. Certaines variables sont très spécifiques à des aspects secondaires de l’enquête et ne sont pas nécessaires pour répondre aux questions de ce projet.

Plutôt que d’utiliser le dataset brut, je réduis le scope à un sous-ensemble de colonnes :

- directement liées au **profil du développeur** (âge, pays, emploi, expérience, formation, type de poste) ;
- liées à la **stack technologique** (langages, bases de données, plateformes, frameworks, autres technologies) ;
- liées à la **rémunération** et au **contexte** (compensation, taille d’entreprise, industrie, usage de l’IA, etc.).

La sélection est basée sur :
- la structure officielle de l’enquête disponible sur le site du Stack Overflow Survey ;
- les besoins de l’analyse à venir (tendances technologiques et profil des développeurs).


In [6]:
col_initial = [
    "ResponseId",
    "MainBranch",
    "Age",
    "RemoteWork",
    "Employment",
    "Country",
    "CodingActivities",
    "EdLevel",
    "LearnCode",
    "LearnCodeOnline",
    "TechDoc",
    "YearsCode",
    "YearsCodePro",
    "DevType",
    "OrgSize",
    "PurchaseInfluence",
    "BuyNewTool",
    "BuildvsBuy",
    "TechEndorse",
    "CompTotal",
    "ConvertedCompYearly",
    "Currency",
    "LanguageHaveWorkedWith",
    "LanguageWantToWorkWith",
    "LanguageAdmired",
    "DatabaseHaveWorkedWith",
    "DatabaseWantToWorkWith",
    "DatabaseAdmired",
    "PlatformHaveWorkedWith",
    "PlatformWantToWorkWith",
    "PlatformAdmired",
    "WebframeHaveWorkedWith",
    "WebframeWantToWorkWith",
    "WebframeAdmired",
    "EmbeddedHaveWorkedWith",
    "EmbeddedWantToWorkWith",
    "EmbeddedAdmired",
    "MiscTechHaveWorkedWith",
    "MiscTechWantToWorkWith",
    "MiscTechAdmired",
    "SOVisitFreq",
    "SOAccount",
    "SOPartFreq",
    "AISelect",
    "AIBen",
    "AIChallenges",
    "JobSat",
    "Industry",
    'OpSysProfessional use', 
    'NEWCollabToolsHaveWorkedWith', 
    'AIToolCurrently Using'
]

df = df[col_initial]
df.shape

(65457, 51)

## 5. Définition des colonnes clé pour identifier un développeur

Chaque ligne correspond à une réponse au questionnaire. Pour éviter de compter plusieurs fois un même profil en cas de duplication, je définis un ensemble de colonnes clé qui caractérisent de manière stable un répondant :

- **MainBranch, DevType** : type de développeur / rôle principal ;
- **Employment, RemoteWork** : contexte d’emploi ;
- **Age, Country, CompTotal** : profil démographique + rémunération ;
- **EdLevel, YearsCode, YearsCodePro** : formation et expérience ;
- **CodingActivities** : nature des activités de code.

La combinaison de ces variables sert de proxy pour identifier un profil unique.  
Les doublons détectés sur ce sous-ensemble sont considérés comme des doublons logiques.


In [7]:
cols_key = [
    "MainBranch",        
    "Age",               
    "Employment",
    "RemoteWork",
    "CodingActivities",
    "EdLevel",
    "YearsCode",
    "YearsCodePro",
    "DevType",
    "Country",
    "CompTotal"
]


## 6. Identification des lignes dupliquées
On recherche maintenant les lignes qui apparaissent comme dupliquées par rapport à cols_key.

In [8]:
# Nombre de doublons basés sur les colonnes clé
nb_duplicated = df.duplicated(subset=cols_key).sum()
print("Nombre de lignes dupliquées basées sur cols_key :", nb_duplicated)

# Aperçu des doublons
df_duplicated = df[df.duplicated(subset=cols_key, keep=False)]
df_duplicated.head()


Nombre de lignes dupliquées basées sur cols_key : 4256


,ResponseId,MainBranch,Age,RemoteWork,Employment,Country,CodingActivities,EdLevel,LearnCode,LearnCodeOnline,...,SOAccount,SOPartFreq,AISelect,AIBen,AIChallenges,JobSat,Industry,OpSysProfessional use,NEWCollabToolsHaveWorkedWith,AIToolCurrently Using
0,1,I am a developer by profession,Under 18 years old,Remote,"Employed, full-time",United States of America,Hobby,Primary/elementary school,Books / Physical media,NaN,...,NaN,NaN,Yes,Increase productivity,NaN,NaN,NaN,NaN,NaN,NaN
1,2,I am a developer by profession,35-44 years old,Remote,"Employed, full-time",United Kingdom of Great Britain and Northern I...,Hobby;Contribute to open-source projects;Other...,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Books / Physical media;Colleague;On the job tr...,Technical documentation;Blogs;Books;Written Tu...,...,Yes,Multiple times per day,"No, and I don't plan to",NaN,NaN,NaN,NaN,MacOS,PyCharm;Visual Studio Code;WebStorm,NaN
2,3,I am a developer by profession,45-54 years old,Remote,"Employed, full-time",United Kingdom of Great Britain and Northern I...,Hobby;Contribute to open-source projects;Other...,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Books / Physical media;Colleague;On the job tr...,Technical documentation;Blogs;Books;Written Tu...,...,Yes,Multiple times per day,"No, and I don't plan to",NaN,NaN,NaN,NaN,Windows,Visual Studio,NaN
3,4,I am learning to code,18-24 years old,NaN,"Student, full-time",Canada,NaN,Some college/university study without earning ...,"Other online resources (e.g., videos, blogs, f...",Stack Overflow;How-to videos;Interactive tutorial,...,No,NaN,Yes,Increase productivity;Greater efficiency;Impro...,Don’t trust the output or answers,NaN,NaN,NaN,NaN,Learning about a codebase;Project planning;Wri...
4,5,I am a developer by profession,18-24 years old,NaN,"Student, full-time",Norway,NaN,"Secondary school (e.g. American high school, G...","Other online resources (e.g., videos, blogs, f...",Technical documentation;Blogs;Written Tutorial...,...,Yes,Multiple times per day,"No, and I don't plan to",NaN,NaN,NaN,NaN,NaN,Vim,NaN


## 7. Suppression des doublons

In [9]:
# Suppression des doublons basés sur les colonnes clé
df_clean = df.drop_duplicates(subset=cols_key, keep="first")

print("Taille avant suppression des doublons :", df.shape)
print("Taille après suppression des doublons :", df_clean.shape)
print("Nombre de lignes supprimées :", df.shape[0] - df_clean.shape[0])


Taille avant suppression des doublons : (65457, 51)
Taille après suppression des doublons : (61201, 51)
Nombre de lignes supprimées : 4256


## 8. Vérification après nettoyage

In [10]:
nb_duplicated_after = df_clean.duplicated(subset=cols_key).sum()
print("Nombre de doublons restants après nettoyage :", nb_duplicated_after)


Nombre de doublons restants après nettoyage : 0


### Observation

Après déduplication, aucune ligne supplémentaire n’est détectée comme doublon sur la base des colonnes clé.  
Le dataset `df_clean` est considéré comme **dédoublonné** pour la suite du pipeline.


## 9. Export du dataset dédoublonné

Le dataset nettoyé des doublons est sauvegardé en tant que nouvelle version :

- **`survey-data_V1_noduplicates.csv`** → point de départ pour le traitement des valeurs manquantes dans le notebook suivant.


In [11]:
df_clean.to_csv("../Data/Survey/processed/survey-data_V1_noduplicates.csv", index=False)

## 10. Résumé

Dans ce module, j’ai :

- réduit le dataset initial à un sous-ensemble de colonnes pertinentes pour l’analyse ;
- défini un jeu de colonnes clé décrivant le profil d’un développeur ;
- identifié et supprimé les doublons logiques sur la base de ces colonnes ;
- généré une version dédoublonnée du dataset (`survey-data_V1_noduplicates.csv`), qui servira d’entrée au notebook suivant dédié au **traitement des valeurs manquantes**.

Cette étape garantit que les analyses ultérieures ne seront pas biaisées par des réponses dupliquées.
